# Evaluate while waiting — advancing with no user message

A conditional edge is normally evaluated at exactly **one** moment: immediately after each
execution of its source node. So a `say_llm` node with `wait_for_user_message=True` parks
indefinitely, and a runtime variable that changes in the background — written by a polling node
in a [companion thread](19_companion_threads.ipynb), say — cannot move the conversation forward
on its own.

`EvaluateWhileWaitingConfig` closes that gap. The caller asks "are my results back?", the
assistant says "let me check", and when the background work lands the workflow advances **by
itself**.

Leaving the config unset (the default) preserves the classic behaviour exactly.

This notebook is about the edge. [Notebook 19](19_companion_threads.ipynb) is about the thread
that changes the variable; you want both.

**See also:** [Evaluate while waiting guide](../docs/guides/waiting_evaluation.md) ·
[`wf_examples/wf_example_progression_24.md`](../wf_examples/wf_example_progression_24.md)

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

## 1. Trigger modes — what *arms* an evaluation

The difference between doing work when something changed and polling blindly.

### `ON_NODE_COMPLETION` (the default, and what you want)

Armed when a nominated node completes another execution, in any thread. Nothing happens until the
node that can actually change the expression's inputs has run.

### `ON_EVERY_BACKGROUND_TICK`

Armed on every background pass. Only for when you genuinely cannot name a trigger node — and it
**requires** a debounce, which the server enforces rather than warns about.

In [ ]:
import _bootstrap  # noqa: F401

import interactly_configs as ic
from interactly import AsyncWorkflowClient, BadRequestError, WorkflowCommand

client = AsyncWorkflowClient()

# The recommended shape.
on_completion = ic.EvaluateWhileWaitingConfig(
    enabled=True,
    trigger_mode=ic.WaitingEvaluationTriggerMode.ON_NODE_COMPLETION,
    trigger_node_logical_ids=["node_placeholder"],
    min_seconds_between_evaluations=1.0,   # optional debounce here
    max_transitions_per_wait=3,            # the default
)

# Blind polling. The debounce is REQUIRED, not advisory.
blind = ic.EvaluateWhileWaitingConfig(
    enabled=True,
    trigger_mode=ic.WaitingEvaluationTriggerMode.ON_EVERY_BACKGROUND_TICK,
    min_seconds_between_evaluations=5.0,
)

print("trigger modes:", [m.value for m in ic.WaitingEvaluationTriggerMode])
print("default max_transitions_per_wait:", on_completion.max_transitions_per_wait)

### Why the debounce is mandatory for blind polling

`ON_EVERY_BACKGROUND_TICK` is always armed. With no debounce it evaluates on every background
pass: the driver loop spins as fast as the event loop allows, each pass appends an event to the
run record, and the runtime is re-serialized on every voice silence tick.

### `max_transitions_per_wait`

Caps how many times this edge may fire during a **single uninterrupted wait**, and resets when a
user message arrives. It guards against a background hop loop in which the caller is never given a
turn — the workflow talking to itself while someone waits on the line.

Note what that means: it bounds one wait, not the run.

## 2. Build a graph that uses it

Same shape as notebook 19: a companion polls, the main thread waits, and the waiting edge reads
**the companion's** variable through the `thread_<id>` prefix.

In [ ]:
POLL_CODE = """
def poll(attempts_so_far):
    try:
        n = int(attempts_so_far)
    except (TypeError, ValueError):
        n = 0
    return n + 1
"""

llm = ic.LLMGroupConfig(llms=[ic.OpenAILLMConfig(model=ic.OPENAIModel.GPT_4_1_MINI, max_tokens=100)])

entry = ic.NoOpNodeConfig(name="Entry", is_start=True)
poller = ic.ToolNodeConfig(
    name="Poller",
    tool_arguments={"attempts_so_far": "[[attempts]]"},
    result_runtime_variable_name="attempts",
    self_loop_config=ic.SelfLoopConfig(enabled=True, max_retries=8, expiry_time=60, time_between_retries=3),
    tool_config=ic.InlinePythonToolConfig(
        name="poll", description="Background poll", signature="Returns the new attempt count.",
        args_schema={"type": "object", "properties": {"attempts_so_far": {"type": "number"}},
                     "required": ["attempts_so_far"]},
        code=POLL_CODE),
)
landed = ic.NoOpNodeConfig(name="Landed", output_runtime_variable_name="landed")
chat = ic.SayLLMNodeConfig(
    name="Chat", wait_for_user_message=True, self_loop=True,
    main_response_config=ic.PromptConfig(prompt="Keep the caller company. ONE short sentence."),
    llms_config=llm,
)
deliver = ic.SayStaticMessageNodeConfig(
    name="Deliver",
    static_messages_config=ic.StaticMessagesConfig(static_messages=["The background work just finished."]),
)

waiting_edge = ic.ConditionalEdgeConfig(
    source_node_logical_id=chat.logical_id,
    destination_node_logical_id=deliver.logical_id,
    condition=ic.ConditionConfig(condition_expression="[[thread_bg.attempts]] >= 3"),
    evaluate_while_waiting_config=ic.EvaluateWhileWaitingConfig(
        enabled=True,
        trigger_node_logical_ids=[poller.logical_id],
    ),
)

edges = [
    ic.DirectEdgeConfig(source_node_logical_id=entry.logical_id, destination_node_logical_id=chat.logical_id),
    ic.DirectEdgeConfig(source_node_logical_id=entry.logical_id, destination_node_logical_id=poller.logical_id,
                        companion_thread_config=ic.CompanionThreadConfig(is_companion_thread=True, thread_id="bg")),
    ic.ConditionalEdgeConfig(source_node_logical_id=poller.logical_id, destination_node_logical_id=landed.logical_id,
                             condition=ic.ConditionConfig(condition_expression="[[attempts]] >= 3")),
    waiting_edge,
]

workflow = await client.workflows.create_from_config(
    ic.WorkflowConfigFullyHydrated(
        workflow_config=ic.WorkflowConfig(name="NB20: Waiting conditions"),
        node_configs=[entry, poller, landed, chat, deliver], edge_configs=edges),
    name="NB20: Waiting conditions")
WORKFLOW_ID = workflow.id
print("Created", WORKFLOW_ID)

## 3. The validation rules

All enforced at save time and returned as **HTTP 400**. Each exists because the alternative is a
conversation that hangs — far harder to diagnose than a rejected save.

| # | Rule | Why |
|---|---|---|
| 1 | Must use `condition_expression` | There has to be something to evaluate. |
| 2 | `condition_freeform` is rejected | A natural-language condition needs an LLM call *per evaluation*. Rejected rather than ignored, so you cannot believe one is being polled. |
| 3 | `ON_NODE_COMPLETION` requires trigger nodes | Otherwise it silently degrades to a blind poll. |
| 4 | Trigger node ids must exist | A typo is a permanent hang. |
| 5 | Trigger nodes must not wait for user input | Their completion is recorded only when they *execute*, not when they park. |
| 6 | `ON_EVERY_BACKGROUND_TICK` requires a debounce > 0 | See section 1. |
| 7 | Not on reverse (global-node) or super-node exit edges | These manipulate the global-node caller stack in ways this path does not yet reproduce. |
| 8 | The source node must wait for user input | Otherwise the config can never fire. |
| 9 | The source node must have no outgoing direct edges | Direct edges take precedence on the normal transition path. |

The cells below trip three of them on purpose, so you can recognise the messages.

In [ ]:
async def expect_rejection(label, edge_list, node_list=None):
    """Try to create a workflow and report the 400 rather than raising."""
    cfg = ic.WorkflowConfigFullyHydrated(
        workflow_config=ic.WorkflowConfig(name=f"NB20 reject: {label}"),
        node_configs=node_list or [entry, poller, landed, chat, deliver],
        edge_configs=edge_list,
    )
    try:
        created = await client.workflows.create_from_config(cfg, name=f"NB20 reject: {label}")
        await client.workflows.delete(created.id)
        print(f"❗ {label}: unexpectedly ACCEPTED")
    except BadRequestError as exc:
        print(f"✅ {label} rejected:\n   {str(exc)[:260]}\n")

base = edges[:3]

# Rule 4 — a trigger node id that does not exist.
await expect_rejection("rule 4 (unknown trigger node)", base + [
    ic.ConditionalEdgeConfig(
        source_node_logical_id=chat.logical_id, destination_node_logical_id=deliver.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[thread_bg.attempts]] >= 3"),
        evaluate_while_waiting_config=ic.EvaluateWhileWaitingConfig(
            enabled=True, trigger_node_logical_ids=["node_does_not_exist"]),
    )])

# Rule 5 — an interactive node as the trigger. `chat` parks rather than completing.
await expect_rejection("rule 5 (interactive trigger)", base + [
    ic.ConditionalEdgeConfig(
        source_node_logical_id=chat.logical_id, destination_node_logical_id=deliver.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[thread_bg.attempts]] >= 3"),
        evaluate_while_waiting_config=ic.EvaluateWhileWaitingConfig(
            enabled=True, trigger_node_logical_ids=[chat.logical_id]),
    )])

# Rule 8 — the source node does not wait for user input, so the config could never fire.
await expect_rejection("rule 8 (source does not wait)", base + [
    ic.ConditionalEdgeConfig(
        source_node_logical_id=landed.logical_id, destination_node_logical_id=deliver.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[thread_bg.attempts]] >= 3"),
        evaluate_while_waiting_config=ic.EvaluateWhileWaitingConfig(
            enabled=True, trigger_node_logical_ids=[poller.logical_id]),
    )])

### Rule 6 is client-side too

`ON_EVERY_BACKGROUND_TICK` with no debounce is refused by the server. Nothing stops you
*constructing* it, so the failure surfaces at save time rather than at authoring time — worth
knowing when the traceback points at `create_from_config` rather than at the config line.

In [ ]:
bad_blind = ic.EvaluateWhileWaitingConfig(
    enabled=True,
    trigger_mode=ic.WaitingEvaluationTriggerMode.ON_EVERY_BACKGROUND_TICK,
    # no min_seconds_between_evaluations
)
print("constructed fine:", bad_blind.trigger_mode, bad_blind.min_seconds_between_evaluations)

await expect_rejection("rule 6 (blind poll, no debounce)", edges[:3] + [
    ic.ConditionalEdgeConfig(
        source_node_logical_id=chat.logical_id, destination_node_logical_id=deliver.logical_id,
        condition=ic.ConditionConfig(condition_expression="[[thread_bg.attempts]] >= 3"),
        evaluate_while_waiting_config=bad_blind,
    )])

## 4. Watching it fire

Two event types tell you what the background evaluator did:

| Type | Meaning |
|---|---|
| `waiting_condition_matched` | The expression became true and the edge was taken, **with no user message**. |
| `waiting_evaluation_boundary` | A parked thread's edges were evaluated. `transitioned` says whether anything moved. |

`waiting_condition_matched` carries the authored `condition_expression`, the
**`hydrated_condition_expression`** as actually evaluated (after variable substitution), which
`trigger_mode` armed it, and `transitions_this_wait`.

When a condition is not firing as expected, **the hydrated expression is the field to look at** —
it shows what the variables actually resolved to.

In [ ]:
async with client.runs.stream(workflow_id=WORKFLOW_ID, command=WorkflowCommand.START) as stream:
    async for event in stream:
        if event.type == "waiting_evaluation_boundary":
            print(f"   evaluated  transitioned={getattr(event, 'transitioned', None)}")

        elif event.type == "waiting_condition_matched":
            print("\n⚡ waiting condition matched — no user message was sent")
            print(f"   authored : {getattr(event, 'condition_expression', None)}")
            print(f"   hydrated : {getattr(event, 'hydrated_condition_expression', None)}")
            print(f"   armed by : {getattr(event, 'trigger_mode', None)}")
            print(f"   this wait: {getattr(event, 'transitions_this_wait', None)}\n")

        elif event.type == "assistant_response":
            print(f"🤖 {event.output}")

        if event.is_terminal():
            print(f"✅ {event.type}")
            break

> **Non-transitioning `waiting_evaluation_boundary` events are not persisted** to the run record.
> They recur on every background pass while a thread is parked, so keeping them would grow the run
> document without bound. They still arrive on the wire — which is why you can see them above but
> will not find them all in `client.runs.get(...)` afterwards.
>
> `ic.should_persist_background_event(event)` mirrors the server's rule if you need to apply it
> yourself.

## 5. Reading the config back

Four defensive accessors read the nested config safely on **any** edge, including edges stored
before the feature existed. They are the right way to inspect a config you did not author.

In [ ]:
for edge in edges:
    if ic.edge_evaluates_while_waiting(edge):
        cfg = ic.edge_waiting_evaluation_config(edge)
        print(f"{edge.type} edge waits:")
        print(f"  trigger_mode             : {cfg.trigger_mode}")
        print(f"  trigger_node_logical_ids : {cfg.trigger_node_logical_ids}")
        print(f"  max_transitions_per_wait : {cfg.max_transitions_per_wait}")
        print(f"  debounce                 : {cfg.min_seconds_between_evaluations}")

## Cleanup

In [ ]:
await client.workflows.delete(WORKFLOW_ID)
await client.close()
print("Deleted", WORKFLOW_ID)

## See also

- [`19_companion_threads.ipynb`](19_companion_threads.ipynb) — the thread that changes the variable
- [Evaluate while waiting guide](../docs/guides/waiting_evaluation.md) · [Self-loops guide](../docs/guides/self_loops.md)

### Gotchas

- **Fork upstream of the parked node** (rule 9). This is the one that catches people out.
- **Freeform conditions are rejected**, not ignored — they would cost an LLM call per evaluation.
- **Do not name an interactive node as a trigger** (rule 5).
- **`ON_EVERY_BACKGROUND_TICK` without a debounce is refused.** Prefer `ON_NODE_COMPLETION`.
- **A REST-driven run needs pumping** or the condition is never evaluated — and see the caveat in
  [notebook 19](19_companion_threads.ipynb) about the REST path today.
- **`max_transitions_per_wait` resets on each user message**, so it bounds one wait, not the run.